# Refusal system 

Decides **before generation** whether a question should be refused. Four methods:

1. **BM25 score**: features from the BM25 scores
2. **Dense score**: features from the cosine scores of Qwen3-Embedding-0.6B
3. **LLM judge**: Qwen3.5-4B is asked whether the retrieved chunks answer the question (P(YES) from the logits)
4. **Own LLM decision (system prompt only)**: the generator says *"I don't have enough information to answer that."*

**Why not `if bm25_score <= 25: refuse`?** A raw BM25 score depends on the question (long questions with rare words score high whether or not the answer exists). On `dataset_100.json` the raw top-1 BM25 score separates answerable from unanswerable questions no better than chance (ROC-AUC ~0.51). So each method turns the scores into query-normalised **features**, a logistic regression combines them into P(unanswerable), and the **cut-off is chosen by 5-fold cross-validation** on the 100 labelled questions (`is_unanswerable`).

Runtime → Change runtime type → **T4 GPU** (only needed for sections 6 and 7; BM25 runs on CPU).

## 1. Setup

In [ ]:
!pip install -q rank_bm25 sentence-transformers scikit-learn

import json
import pickle
import re
import sqlite3
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# Files used by the scripts (same as in Script Resources/).
# Option A (active): clone the repository and read Script Resources/ directly
!git clone -q https://github.com/cedricfyc/nlp-tasks-2026.git /content/nlp-tasks-2026
RESOURCES = Path("/content/nlp-tasks-2026/Assignment 1/Python Scripts/Script Resources")

# Option B: Google Drive (comment out Option A and adjust the folder name)
# from google.colab import drive
# drive.mount('/content/drive')
# RESOURCES = Path("/content/drive/MyDrive/MAI NLP WS26_27")

DB_PATH = RESOURCES / "student_assistant_rag_0_6B.db"
BM25_CACHE = RESOURCES / "bm25_chunks.pkl"
DATASET_PATH = RESOURCES / "dataset_100.json"
OUTPUT_DIR = Path("/content/refusal_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
MODEL_ID = "Qwen/Qwen3.5-4B"
MODEL_PATH = "/content/drive/MyDrive/models/Qwen3.5-4B"   # used in section 7
TOP_K = 3              # chunks per search method (as in the RAG pipeline)
FEATURE_DEPTH = 5      # chunks the features look at (more than TOP_K so a "gap" can be measured)
EMBED_TYPE = np.float32
REFUSAL_MESSAGE = "I don't have enough information to answer that."

questions = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
print(len(questions), "questions,", sum(q["is_unanswerable"] for q in questions), "unanswerable")

## 2. Data helpers (database, BM25, embeddings)

In [ ]:
def connect_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def bm25_tokenise(text):
    """Unicode-aware tokeniser preserving words and hyphenated terms (same as the pipeline)."""
    return re.findall(r"\b[\w-]+\b", str(text).lower(), flags=re.UNICODE)


def load_bm25(cache_path=BM25_CACHE):
    with Path(cache_path).open("rb") as f:
        payload = pickle.load(f)
    print(f"Loaded BM25 index with {len(payload['chunk_ids']):,} chunks.")
    return payload


def load_embedding_matrix():
    """All chunk embeddings as a row-normalised matrix. Returns (chunk_ids, matrix)."""
    with connect_db() as conn:
        rows = conn.execute("SELECT chunk_id, embedding FROM chunks "
                            "WHERE embedding IS NOT NULL ORDER BY chunk_id").fetchall()
    ids = [int(r["chunk_id"]) for r in rows]
    matrix = np.vstack([np.frombuffer(r["embedding"], dtype=EMBED_TYPE) for r in rows])
    return ids, matrix / np.maximum(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-12)


def fetch_texts(chunk_ids):
    """Chunk texts in the order of chunk_ids."""
    with connect_db() as conn:
        rows = conn.execute(f"SELECT chunk_id, text FROM chunks WHERE chunk_id IN "
                            f"({','.join('?' * len(chunk_ids))})", chunk_ids).fetchall()
    texts = {int(r["chunk_id"]): r["text"] for r in rows}
    return pd.DataFrame({"chunk_id": chunk_ids, "text": [texts[i] for i in chunk_ids]})

## 3. Features: BM25 (method 1) and dense (method 2)

In [ ]:
BM25_FEATURES = ["norm_top1", "coverage_top1", "coverage_topk", "relative_gap", "log_top1"]


def bm25_features(query: str, bm25_payload, depth: int = FEATURE_DEPTH) -> dict:
    """Query-normalised features from the BM25 scores over the whole corpus."""
    bm25 = bm25_payload["bm25"]
    tokens = bm25_tokenise(query)
    scores = np.asarray(bm25.get_scores(tokens))
    order = np.argsort(scores)[::-1][:depth]
    top = scores[order]

    # IDF of each distinct query word (negative IDF = very common word -> ignore)
    idf = {t: max(float(bm25.idf.get(t, 0.0)), 0.0) for t in set(tokens)}
    idf_sum = sum(idf.values()) or 1e-9
    # Highest BM25 score this query could reach: every word with saturated term frequency
    max_possible = idf_sum * (bm25.k1 + 1)

    def coverage(doc_indices) -> float:
        """IDF-weighted share of query words that appear in these chunks."""
        found = set()
        for i in doc_indices:
            found |= {t for t in idf if t in bm25.doc_freqs[i]}
        return sum(idf[t] for t in found) / idf_sum

    top1 = float(top[0]) if top.size else 0.0
    return {
        "norm_top1": top1 / max_possible,
        "coverage_top1": coverage(order[:1]),
        "coverage_topk": coverage(order[:TOP_K]),
        "relative_gap": (top1 - float(top[-1])) / top1 if top1 > 0 else 0.0,
        "log_top1": float(np.log1p(top1)),
    }


DENSE_FEATURES = ["top1", "mean_topk", "gap_1_5", "z_top1"]


def dense_features(query: str, vector_matrix, embedding_model, depth: int = FEATURE_DEPTH) -> dict:
    """Features from the cosine similarities between the query and every chunk."""
    q = embedding_model.encode([query], prompt_name="query", normalize_embeddings=True,
                               convert_to_numpy=True)[0].astype(np.float32)
    scores = vector_matrix @ q
    top = np.sort(scores)[::-1][:depth]
    return {
        "top1": float(top[0]),
        "mean_topk": float(top[:TOP_K].mean()),
        "gap_1_5": float(top[0] - top[-1]),
        # how far the best chunk stands out from the corpus as a whole
        "z_top1": float((top[0] - scores.mean()) / (scores.std() + 1e-9)),
    }

## 4. Calibrated cut-off (used by methods 1, 2 and 3)

In [ ]:
@dataclass
class RefusalDecision:
    refuse: bool
    method: str
    p_unanswerable: float          # 0..1, higher = more likely to refuse
    threshold: float
    features: dict = field(default_factory=dict)


def choose_threshold(p, y, max_false_refusal=0.10):
    """Cut-off on P(unanswerable) with the best balanced accuracy, refusing at most
    max_false_refusal of the answerable questions."""
    p, y = np.asarray(p), np.asarray(y)
    best = None
    for t in np.unique(np.concatenate([p, [1.0]])):
        refuse = p >= t
        recall = refuse[y == 1].mean()              # unanswerable questions refused
        false_refusal = refuse[y == 0].mean()       # answerable questions refused
        bal_acc = (recall + (1 - false_refusal)) / 2
        if false_refusal <= max_false_refusal and (best is None or bal_acc > best[1]):
            best = (float(t), bal_acc, recall, false_refusal)
    t, bal_acc, recall, false_refusal = best
    return t, {"threshold": t, "balanced_accuracy": float(bal_acc),
               "refusal_recall": float(recall), "false_refusal_rate": float(false_refusal),
               "max_false_refusal": max_false_refusal}


class ScoreRefuser:
    """Logistic regression on the features + a cut-off chosen by cross-validation."""

    def __init__(self, method, feature_names, model, threshold, metrics):
        self.method, self.feature_names = method, feature_names
        self.model, self.threshold, self.metrics = model, threshold, metrics

    @classmethod
    def fit(cls, method, feature_names, X, y, max_false_refusal=0.10, folds=5, seed=0):
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import roc_auc_score
        from sklearn.model_selection import StratifiedKFold, cross_val_predict
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler

        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=int)

        def model():
            return make_pipeline(StandardScaler(),
                                 LogisticRegression(class_weight="balanced", C=1.0))

        cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
        oof = cross_val_predict(model(), X, y, cv=cv, method="predict_proba")[:, 1]
        threshold, stats = choose_threshold(oof, y, max_false_refusal)
        stats["cv_roc_auc"] = float(roc_auc_score(y, oof))
        stats["n_questions"], stats["n_unanswerable"] = int(len(y)), int(y.sum())
        return cls(method, feature_names, model().fit(X, y), threshold, stats), oof

    def decide(self, features: dict) -> RefusalDecision:
        x = np.array([[features[name] for name in self.feature_names]], dtype=float)
        p = float(self.model.predict_proba(x)[0, 1])
        return RefusalDecision(p >= self.threshold, self.method, p, self.threshold, features)


def calibrate(method, feature_names, features_of, max_false_refusal=0.10):
    """Features for every question -> fit -> print the cross-validated results."""
    rows = []
    for i, item in enumerate(questions, start=1):
        rows.append({"id": item["id"], "type": item["type"],
                     "is_unanswerable": int(item["is_unanswerable"]),
                     "question": item["question"], **features_of(item["question"])})
        if i % 20 == 0:
            print(f"features: {i}/{len(questions)}")
    df = pd.DataFrame(rows)

    refuser, oof = ScoreRefuser.fit(method, feature_names, df[feature_names].values,
                                    df["is_unanswerable"].values, max_false_refusal)
    df["p_unanswerable_cv"] = oof
    df["refused_cv"] = oof >= refuser.threshold
    df.to_csv(OUTPUT_DIR / f"refusal_{method}_questions.csv", index=False)

    m = refuser.metrics
    print(f"\n=== {method} refusal (5-fold cross-validated) ===")
    print(f"questions: {m['n_questions']} ({m['n_unanswerable']} unanswerable)")
    print(f"ROC-AUC:                     {m['cv_roc_auc']:.3f}   (0.5 = chance)")
    print(f"cut-off P(unanswerable) >=   {refuser.threshold:.3f}")
    print(f"unanswerable refused:        {m['refusal_recall']:.0%}")
    print(f"answerable wrongly refused:  {m['false_refusal_rate']:.0%}  "
          f"(limit {m['max_false_refusal']:.0%})")
    print(f"balanced accuracy:           {m['balanced_accuracy']:.3f}")
    print("\nRefusal rate by question type:")
    print(df.groupby("type")["refused_cv"].mean().map("{:.0%}".format).to_string())
    return refuser, df


results = {}   # method -> metrics, for the comparison table at the end

## 5. Method 1: BM25 (CPU)

In [ ]:
bm25_payload = load_bm25()
bm25_refuser, bm25_df = calibrate("bm25", BM25_FEATURES, lambda q: bm25_features(q, bm25_payload))
results["bm25"] = bm25_refuser.metrics

## 6. Method 2: Dense score (GPU recommended)

In [ ]:
from sentence_transformers import SentenceTransformer

vector_chunk_ids, vector_matrix = load_embedding_matrix()
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
dense_refuser, dense_df = calibrate(
    "dense", DENSE_FEATURES, lambda q: dense_features(q, vector_matrix, embedding_model))
results["dense"] = dense_refuser.metrics

## 7. Method 3: LLM judge (GPU)

The retrieved chunks are the same as in the RAG pipeline: BM25 top-3 + dense top-3, merged with Reciprocal Rank Fusion. Needs section 6 (embedding model) to have run.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=dtype, device_map="auto",
                                             local_files_only=True, trust_remote_code=True).eval()

In [ ]:
def retrieve(query, top_k=TOP_K, rrf_k=60):
    """BM25 top-k + dense top-k merged with Reciprocal Rank Fusion (as in the pipeline)."""
    bm25_scores = np.asarray(bm25_payload["bm25"].get_scores(bm25_tokenise(query)))
    lexical = [bm25_payload["chunk_ids"][i] for i in np.argsort(bm25_scores)[::-1][:top_k]]
    q = embedding_model.encode([query], prompt_name="query", normalize_embeddings=True,
                               convert_to_numpy=True)[0].astype(np.float32)
    dense_scores = vector_matrix @ q
    semantic = [vector_chunk_ids[i] for i in np.argsort(dense_scores)[::-1][:top_k]]
    rrf = {}
    for ranked in (lexical, semantic):
        for rank, chunk_id in enumerate(ranked):
            rrf[chunk_id] = rrf.get(chunk_id, 0) + 1 / (rrf_k + rank + 1)
    return fetch_texts(sorted(rrf, key=rrf.get, reverse=True))


JUDGE_PROMPT = """You check a retrieval system for international students in Germany.
Decide whether the SOURCES contain the information needed to answer the QUESTION.
Answer NO if the question asks for personal or private data (the user's own account,
application, contract, appointment, medical record, a decision about their individual
case) or anything the sources do not state. Otherwise answer YES.
Reply with one word: YES or NO.

QUESTION:
{question}

SOURCES:
{sources}"""


class LLMJudge:
    """Asks the LLM 'can these chunks answer the question?' and reads P(YES) from the logits."""

    def __init__(self, tokenizer, model, max_chars_per_chunk=1200):
        self.tokenizer, self.model = tokenizer, model
        self.max_chars = max_chars_per_chunk
        self.yes_id = tokenizer.encode("YES", add_special_tokens=False)[0]
        self.no_id = tokenizer.encode("NO", add_special_tokens=False)[0]

    def p_answerable(self, question: str, context_df) -> float:
        sources = "\n\n".join(f"[{i}] {str(text)[:self.max_chars]}"
                               for i, text in enumerate(context_df["text"], start=1))
        messages = [{"role": "user",
                     "content": JUDGE_PROMPT.format(question=question, sources=sources)}]
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = self.tokenizer([prompt], return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, [self.yes_id, self.no_id]].float()
        return float(torch.softmax(logits, dim=0)[0])


LLM_JUDGE_FEATURES = ["p_yes"]
judge = LLMJudge(tokenizer, model)
judge_refuser, judge_df = calibrate(
    "llm_judge", LLM_JUDGE_FEATURES, lambda q: {"p_yes": judge.p_answerable(q, retrieve(q))})
results["llm_judge"] = judge_refuser.metrics

## 8. Method 4: Own LLM decision (system prompt only)

No extra model: the generator's system prompt already says *"If the answer is unavailable, say exactly: I don't have enough information to answer that."* This cell detects that sentence. To measure its refusal rate, run the RAG answers for the 100 questions (e.g. `Test Scripts/rag_answer_100_questions.ipynb`) and apply `is_self_refusal` to each answer.

In [ ]:
_SELF_REFUSAL = re.compile(r"(?:don't|do not) have enough information", re.IGNORECASE)


def is_self_refusal(answer: str) -> bool:
    """True if the generator refused by itself (system prompt rule)."""
    return bool(_SELF_REFUSAL.search(answer or ""))


print(is_self_refusal(REFUSAL_MESSAGE), is_self_refusal("The fee is 58 euros per semester [1]."))

## 9. Comparison of the methods

In [ ]:
pd.DataFrame(results).T[["cv_roc_auc", "refusal_recall", "false_refusal_rate",
                         "balanced_accuracy", "threshold"]].round(3)

## 10. Try one question

In [ ]:
question = "How much money is currently available in my blocked account?"
print(bm25_refuser.decide(bm25_features(question, bm25_payload)))
# print(dense_refuser.decide(dense_features(question, vector_matrix, embedding_model)))
# print(judge_refuser.decide({"p_yes": judge.p_answerable(question, retrieve(question))}))